# Lesson 07: Three-way reconciliation (platform vs bridge vs LP)
**Level:** advanced · **Before this:** Lessons 02 and 05 help, but every term is explained again here.

> **How to read this notebook:** you don't need to understand the Python. Read the 🔧 line above each grey code
> box (what the code does, in plain words), then the output and the explanation under it.

### What this lesson is about
**Reconciliation** ("recon") = comparing two records that should agree, and explaining every difference. Like
checking your bank statement against your own list of payments.

Every **A-book** trade (a client trade the broker passes on to an LP, a liquidity provider) is recorded in **three**
places:

| # | Record | What it is | File |
|---|---|---|---|
| 1 | **Platform deals** | what the **client** was given (his position) | `data/platform/deals.csv` |
| 2 | **Bridge FIX fills** | what the LP told us in real time: `35=8` messages (MsgType = ExecutionReport) with `150=F` (ExecType = Trade) | `data/fix_logs/` |
| 3 | **LP statement** | the LP's own list of trades, sent at end of day. This is what the LP will settle | `data/lp_statements/` |

If they don't agree, the broker's **hedge** (its positions at the LPs) doesn't match its **client positions**, so it
carries market risk **it doesn't know about**. This lesson matches all three, trade by trade, and finds three
**breaks** (differences) left by earlier incidents:

| Incident | Where you first saw it | What break it leaves |
|---|---|---|
| **S05** | Lesson 02: LP_B re-sent a fill (with `43=Y`, PossDupFlag = "possible copy") after a wrong ResendRequest | the same fill booked **twice** on the platform |
| **S06** | Lessons 02 and 05: LP_C's connection died; 3 orders never got an answer | a fill LP_C made but **we never received** |
| **S10** | Lessons 01 and 05: gold sent to LP_C in lots instead of ounces | client trades **100 times bigger** than the LP trades |

### Words used in this lesson
| Word | Meaning |
|---|---|
| **ExecID** | Tag `17` in FIX: the LP's unique reference for one execution (one fill). E.g. `LP_B-E000075` |
| **ClOrdID** | Tag `11`: the bridge's order reference `BR<client order>-<attempt>` |
| **Lot** | 1 lot EURUSD/GBPUSD = 100,000 units; 1 lot gold = 100 oz (the **contract size**) |
| **Break** | A difference between two records that should agree |
| **Gross / net position** | Gross = total traded (buys + sells). Net = buys − sells = what is still open |
| **Notional** | Size in money: lots × contract size × price |

🔧 **The code below** loads the three records and counts them: A-book deals on the platform, fill messages in the FIX
logs (and how many of those are re-sent copies with `43=Y`), and trades on the LP statements.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from bridgelab import analysis as A
pd.set_option("display.width", 200, "display.max_columns", 30)

msgs = A.load_fix_messages()
quotes = A.lp_quotes(msgs)
orders, deals = A.load_platform()
lp_stmt = A.load_lp_statements()
sym, cfg, audit, clients = A.load_reference()
CONTRACT = sym.contract_size.to_dict()

platform = deals[deals.book == "A"].copy()
bridge = msgs[(msgs.msg_type == "8") & (msgs.exec_type == "F")].copy()
print(f"platform A-book deals: {len(platform)} | bridge fill reports: {len(bridge)} "
      f"(of which PossDup re-sends: {(bridge.poss_dup == 'Y').sum()}) | LP statement trades: {len(lp_stmt)}")

platform A-book deals: 745 | bridge fill reports: 745 (of which PossDup re-sends: 1) | LP statement trades: 745


**745 / 745 / 745.** The three counts are identical, and that's the trap: **equal counts don't mean the records
agree.** One of the 745 bridge messages is a re-sent copy (`43=Y`), so the bridge really has 744 distinct fills. The
breaks only appear when you match **line by line**.

---
## 1. Choose the matching key and the common unit

Two decisions before matching anything:

**The key (what identifies the same trade in two records):** the LP's **ExecID** (tag `17`). It's unique for every
execution, it's in the FIX message, the platform stores it with each deal (column `lp_exec_id`), and the LP statement
has it too. Never match on price and time alone: two trades can easily have the same price and time.

**The unit (how to compare sizes):** the platform counts in **lots**, the FIX messages and LP statements in **units**
(EUR, ounces). Convert everything to lots using the **official** contract size (100,000 for currencies, **100 oz for
gold**), from the symbol specifications. **Don't** use the bridge's own conversion setting: if that setting is wrong
(it is, for LP_C gold), it would make the wrong numbers agree and hide the error.

🔧 **The code below** converts the FIX fill quantities (tag 32, LastQty) and the LP statement quantities to lots with
the official contract size, and drops the re-sent copies (`43=Y`) from the bridge list, so each real execution is
counted once.

In [2]:
bridge["lots"] = bridge.last_qty / bridge.symbol.map(CONTRACT)
lp_stmt["lots"] = lp_stmt.qty / lp_stmt.symbol.map(CONTRACT)
bridge_u = bridge[bridge.poss_dup != "Y"]            # one row per real execution

---
## 2. Recon 1: bridge (FIX) vs LP statement

*Did we receive every trade the LP says it did, with the same quantity and price?*

🔧 **The code below** matches every bridge fill with every LP statement line on (LP, ExecID) and gives each one a
status: **matched**; **missing at LP** (we have it, the LP doesn't); **missing at bridge** (the LP has it, we don't);
**quantity differs**; **price differs**. Then it counts the statuses per LP.

In [3]:
r1 = bridge_u[["lp", "exec_id", "cl_ord_id", "symbol", "side", "lots", "last_px"]].merge(
    lp_stmt[["lp", "exec_id", "lots", "price", "trade_time"]], on=["lp", "exec_id"], how="outer",
    suffixes=("_bridge", "_lp"), indicator=True)
r1["status"] = r1._merge.astype(str).map({"both": "matched", "left_only": "missing at LP", "right_only": "missing at bridge"})
r1.loc[(r1.status == "matched") & ~np.isclose(r1.lots_bridge, r1.lots_lp), "status"] = "quantity differs"
r1.loc[(r1.status == "matched") & ~np.isclose(r1.last_px, r1.price), "status"] = "price differs"
r1.groupby(["lp", "status"]).size().unstack(fill_value=0)

status,matched,missing at bridge
lp,,
LP_A,389,0
LP_B,119,0
LP_C,236,1


**Reading the table:** LP_A (389) and LP_B (119) match perfectly. LP_C: 236 matched, but **1 trade is on LP_C's
statement that we never received.**

(Why doesn't the LP_C gold quantity problem show up here? Because the bridge and LP_C **agree with each other**: both
say the small quantity was traded. The problem is between them and the **platform**, in section 3.)

🔧 **The code below** shows that missing trade.

In [4]:
r1[r1.status == "missing at bridge"]

,lp,exec_id,cl_ord_id,symbol,side,lots_bridge,last_px,lots_lp,price,trade_time,_merge,status
696,LP_C,LP_C-E000189,NaN,NaN,NaN,NaN,NaN,0.03,4096.8,2025-10-15 13:00:00.089,right_only,missing at bridge


**ExecID `LP_C-E000189`:** LP_C says it traded **0.03 lots** (3 oz) of gold at **4096.80** at **13:00:00.089**. The
bridge columns are empty: no FIX message about it ever reached us.

🔧 **The code below** uses the ClOrdID (tag 11) on LP_C's statement to find the client order this trade belongs to,
and shows what the platform did with it.

In [5]:
lost = r1[r1.status == "missing at bridge"].merge(lp_stmt[["exec_id", "cl_ord_id", "side", "symbol"]], on="exec_id",
                                                  suffixes=("", "_stmt"))
lost["order_id"] = lost.cl_ord_id_stmt.str.extract(r"BR(\d+)-")[0].astype(int)
lost[["exec_id", "cl_ord_id_stmt", "symbol_stmt", "side_stmt", "lots_lp", "price", "trade_time"]].merge(
    orders[["order_id", "client_id", "lots", "status", "reject_reason"]], left_on=lost.order_id, right_on="order_id")

,exec_id,cl_ord_id_stmt,symbol_stmt,side_stmt,lots_lp,price,trade_time,order_id,client_id,lots,status,reject_reason
0,LP_C-E000189,BR701128-1,XAUUSD,SELL,0.03,4096.8,2025-10-15 13:00:00.089,701128,C1003,3.0,REJECTED,LP timeout


**Break type 1: an execution we never received (incident S06).**

It's order **BR701128-1**, the first of the three "no answer" orders from Lessons 02 and 05. The full story:

| Time | Event |
|---|---|
| 12:59:59.953 | Bridge sends the order to LP_C: client C1003 sells 3 lots of gold |
| 13:00:00 | LP_C's connection goes silent |
| 13:00:00.089 | **LP_C fills it** (according to its own statement), but its `35=8` (ExecutionReport) never reaches us |
| ~13:00:01 | Bridge gives up waiting; the platform tells client C1003 **"REJECTED, LP timeout"** |
| 13:00:22 | Bridge hangs up (`35=5`, Logout) |
| 13:02:30 | Reconnect with `141=Y` (ResetSeqNumFlag = "both restart message numbers at 1") |

Normally, after a reconnect, the bridge would see a gap in LP_C's message numbers and ask for the missing messages
(`35=2`, ResendRequest), and LP_C would re-send the fill. But **`141=Y` restarted the numbering at 1**, so there was no
gap to detect, and the lost fill was simply forgotten.

**Result:** the broker has a **short gold position at LP_C that no client position matches**. The client was told his
order was rejected, so he has nothing. The broker is now holding a trade nobody decided to take.

And notice the size: the client asked for **3 lots** (300 oz) and LP_C executed **0.03 lots** (3 oz). That's the
gold quantity bug (S10) as well: **two bugs in one trade.**

---
## 3. Recon 2: platform deals vs bridge fills

*Did the platform book each fill exactly once, with the right size?*

🔧 **The code below** matches platform A-book deals with bridge fills on ExecID (`lp_exec_id` on the platform = tag 17
in FIX), then lists any platform deals that **share the same ExecID**: there should be none, since each LP execution
should give exactly one client deal.

In [6]:
r2 = platform[["deal_id", "order_id", "client_id", "lp", "lp_exec_id", "symbol", "side", "lots", "time"]].merge(
    bridge_u[["exec_id", "lots"]].rename(columns={"exec_id": "lp_exec_id", "lots": "lots_bridge"}),
    on="lp_exec_id", how="outer", indicator=True)
dupes = r2[r2.duplicated("lp_exec_id", keep=False) & r2.lp_exec_id.notna()]
print("platform deals sharing the same LP ExecID:")
dupes

platform deals sharing the same LP ExecID:


,deal_id,order_id,client_id,lp,lp_exec_id,symbol,side,lots,time,lots_bridge,_merge
463,901028,700965,C1001,LP_B,LP_B-E000075,EURUSD,BUY,2.0,2025-10-15 12:45:09.833,2.0,both
464,901029,700965,C1001,LP_B,LP_B-E000075,EURUSD,BUY,2.0,2025-10-15 12:45:10.016,2.0,both


**Break type 2: duplicate booking (incident S05).**

Deals **901028** (12:45:09.833) and **901029** (12:45:10.016): both for client order 700965, client C1001, buy 2 lots
EURUSD, **same ExecID `LP_B-E000075`**.

This is the story from Lesson 02:
1. 12:45:09.831: LP_B's fill arrives normally → the platform books deal 901028. Correct.
2. 12:45:10.178: the bridge notices lost messages and sends a ResendRequest (`35=2`), but starts **3 numbers too
   early**, asking again for messages it already had.
3. 12:45:10.192: LP_B correctly re-sends the fill, marked `43=Y` (PossDupFlag = "this may be a copy").
4. The bridge doesn't check the ExecID, passes it on, and the platform books it **again** → deal 901029.

**Result:** client C1001 now has **4 lots** long EURUSD instead of 2. The broker's hedge at LP_B is still 2 lots. The
broker is **short 2 lots (≈ $233,000) unhedged** against this client, and the client has a position he never traded.

🔧 **The code below** finds every deal where the platform's lots and the bridge's lots (converted with the **official**
contract size) are different, and groups them by LP and symbol.

In [7]:
qty_breaks = r2[(r2._merge == "both") & ~np.isclose(r2.lots, r2.lots_bridge)]
print(f"platform vs bridge quantity mismatches: {len(qty_breaks)}")
qty_breaks.groupby(["lp", "symbol"]).agg(deals=("deal_id", "size"), platform_lots=("lots", "sum"),
                                         bridge_lots=("lots_bridge", "sum")).round(2)

platform vs bridge quantity mismatches: 97


,,deals,platform_lots,bridge_lots
lp,symbol,,,
LP_C,XAUUSD,97,85.21,0.85


**Break type 3: quantity mismatch (incident S10).**

**97 deals, all LP_C gold.** The platform gave clients a total of **85.21 lots**; LP_C actually executed **0.85 lots**,
exactly 100 times less.

**Why nobody noticed in real time:** the bridge setting `LP_C:XAUUSD = 1` was used **both** to send the order (lots ×
1 = "ounces") and to read the fill back (ounces ÷ 1 = "lots"). So inside the bridge, everything "agreed": the client
wanted 3, LP_C filled 3. Only a reconciliation in an **independent** unit (the official 100 oz per lot) shows that
3 oz is not 3 lots.

---
## 4. Position reconciliation by LP and symbol

Trade-by-trade recon tells you **why** things differ. Position recon tells you **how much risk** you're carrying
right now: for each LP and symbol, compare the **net position** (buys − sells) according to the platform (what we
believe we're hedged) with the net position according to the LP statement (what we really hold).

🔧 **The code below** calculates, for each LP and symbol: gross lots on each side, net lots on each side, the
difference (`break_lots`), and the difference in USD (break lots × contract size × last market price).

In [8]:
sgn = lambda side: np.where(side == "BUY", 1, -1)
plat_pos = platform.assign(net=platform.lots * sgn(platform.side)).groupby(["lp", "symbol"]).net.sum()
lp_pos = lp_stmt.assign(net=lp_stmt.lots * sgn(lp_stmt.side)).groupby(["lp", "symbol"]).net.sum()
pos = pd.DataFrame({"platform_gross_lots": platform.groupby(["lp", "symbol"]).lots.sum(),
                    "lp_gross_lots": lp_stmt.groupby(["lp", "symbol"]).lots.sum(),
                    "platform_net_lots": plat_pos, "lp_net_lots": lp_pos}).fillna(0)
pos["break_lots"] = pos.platform_net_lots - pos.lp_net_lots
mid_last = {s: A.mid_series(quotes, s).iloc[-1] for s in sym.index}
pos["break_usd_notional"] = [b * CONTRACT[s] * mid_last[s] for (lp, s), b in zip(pos.index, pos.break_lots)]
pos.round(2)

platform_gross_lots  lp_gross_lots  platform_net_lots  lp_net_lots  break_lots  break_usd_notional
lp   symbol                                                                                                    
LP_A EURUSD               375.14         375.14             -48.32       -48.32        0.00                0.00
     GBPUSD                81.45          81.45              14.35        14.35        0.00                0.00
     XAUUSD               307.14         307.14              42.78        42.78        0.00                0.00
LP_B EURUSD                67.19          65.19              14.37        12.37        2.00           233074.00
     GBPUSD                17.05          17.05              -6.77        -6.77        0.00                0.00
     XAUUSD               113.19         113.19              -9.31        -9.31       -0.00               -0.00
LP_C EURUSD               195.01         195.01              32.69        32.69        0.00                0.00
     GBPUSD                79.15          79.15             -10.67       -10.67        0.00                0.00
     XAUUSD                85.21           0.88              -2.13        -0.05       -2.08          -850818.79

**Reading the table:** 7 of the 9 lines show a break of 0: perfect. Two don't:

| LP / symbol | Platform net | LP net | Break | In USD | Cause |
|---|---|---|---|---|---|
| **LP_B / EURUSD** | +14.37 lots | +12.37 lots | **+2.00 lots** | **≈ $233,000** | the duplicate deal (S05) |
| **LP_C / XAUUSD** | −2.13 lots | −0.05 lots | **−2.08 lots** | **≈ −$851,000** | the quantity bug (S10) + the lost fill (S06) |

* **LP_C gold:** look at the **gross** columns first: the platform booked **85.21 lots** of gold against LP_C, LP_C
  executed **0.88 lots**. Clients both bought and sold, so most of it cancelled out in the net, but the remaining net
  break is still **about 2 lots ≈ $850,000 of gold** that the broker believes is hedged and isn't. If gold moves by $10,
  that's about $2,000 of profit or loss that nobody decided to take, and it changes with every new LP_C gold trade.
* The lost S06 fill (−0.03 lots) is included in LP_C's −0.05 net: the LP holds it, the platform doesn't.

---
## 5. Break register

Every break goes into a **break register**: one line per break, with an ID, cause, action and (in real life) an owner,
a deadline and a status. Open breaks are reviewed every day until closed; old open breaks are an audit finding.

🔧 **The code below** writes the three breaks into a register table.

In [9]:
register = pd.DataFrame([
    dict(break_id="RB-1", scenario="S06", type="Execution missing at bridge/platform",
         where="LP_C statement vs FIX", detail=f"{len(lost)} LP_C fill(s) at ~13:00 never received; client order rejected 'LP timeout'",
         root_cause="Session died with the ExecutionReport in flight; reconnect used ResetSeqNumFlag=Y, so no resend",
         action="Close or transfer the orphan LP position; never reset sequence numbers after an unclean disconnect; "
                "treat timeouts as 'unknown' and query order status (35=H) on reconnect"),
    dict(break_id="RB-2", scenario="S05", type="Duplicate deal", where="platform vs FIX",
         detail=f"ExecID {dupes.lp_exec_id.iloc[0]} booked {len(dupes)} times",
         root_cause="ResendRequest from the wrong sequence number + PossDup (43=Y) not de-duplicated on ExecID",
         action="Cancel the duplicate client deal (with client notice); de-duplicate on ExecID; fix BeginSeqNo"),
    dict(break_id="RB-3", scenario="S10", type="Quantity mismatch", where="platform vs FIX vs LP (in lots)",
         detail=f"{len(qty_breaks)} LP_C gold trades executed in ounces = lots x 1",
         root_cause="lp_quantity_multiplier LP_C:XAUUSD = 1 set at onboarding (CHG-2291)",
         action="Hedge the open gap now; fix the multiplier to 100; pre-trade check qty/lots = contract size; "
                "daily position recon per LP"),
])
register

,break_id,scenario,type,where,detail,root_cause,action
0,RB-1,S06,Execution missing at bridge/platform,LP_C statement vs FIX,1 LP_C fill(s) at ~13:00 never received; clien...,Session died with the ExecutionReport in fligh...,Close or transfer the orphan LP position; neve...
1,RB-2,S05,Duplicate deal,platform vs FIX,ExecID LP_B-E000075 booked 2 times,ResendRequest from the wrong sequence number +...,Cancel the duplicate client deal (with client ...
2,RB-3,S10,Quantity mismatch,platform vs FIX vs LP (in lots),97 LP_C gold trades executed in ounces = lots x 1,lp_quantity_multiplier LP_C:XAUUSD = 1 set at ...,Hedge the open gap now; fix the multiplier to ...


**The register in plain words:**

| Break | What | Why | What to do |
|---|---|---|---|
| **RB-1** (S06) | LP_C filled an order we think was rejected | the connection died while the fill was on its way, and reconnecting with `141=Y` (ResetSeqNumFlag: restart numbering at 1) meant it was never re-sent | close or transfer the leftover LP position; don't reset numbering after a crash; after a reconnect, ask the LP for the status of every unanswered order (`35=H`, MsgType = OrderStatusRequest) |
| **RB-2** (S05) | one fill booked twice | wrong ResendRequest + the copy (`43=Y`, PossDupFlag) not checked against its ExecID (tag 17) | cancel the duplicate deal (and tell the client); check ExecID before booking; fix the ResendRequest start number |
| **RB-3** (S10) | LP_C gold trades 100× too small | wrong multiplier (1 instead of 100) set during LP_C's onboarding, change ticket CHG-2291 | hedge the open gap now; set the multiplier to 100; add a check before each order; reconcile positions per LP every day |

---
## Reconciliation checklist (daily)
1. Match **bridge fills ↔ LP statements** on ExecID (tag 17) → missing or extra trades, different prices or quantities.
2. Match **platform deals ↔ bridge fills** on ExecID → duplicates, orphans, quantity conversion errors.
3. Compare **net positions per LP and symbol** in an independent unit (lots via the official contract size).
4. Put a USD value on every break at the current price, and escalate anything above a set limit.
5. Chase unanswered orders (timeouts) with the LP **the same day**: they are unknowns, not rejects.

**In real life** this runs every morning, in Excel, SQL or a reconciliation tool (e.g. Duco, SmartStream TLM, or an
in-house system), on the LP statements that arrive overnight.

**Next:** Lesson 08 turns every check from Lessons 02–07 into an automatic monitor and an incident playbook.